# Code Refactoring LLM Training with CodeT5This notebook trains a CodeT5 model to perform code refactoring using the CodeXGLUE code refinement dataset.## Overview- **Dataset**: google/code_x_glue_cc_code_refinement- **Base Model**: CodeT5-base- **Task**: Transform buggy code into refactored/fixed code- **Components**: Data loader, Tokenizer, Preprocessor, Model Architecture, Training, Evaluation classes

## 1. Environment Setup and Dependencies

In [ ]:
# Install required packages
!pip install transformers datasets torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install accelerate evaluate rouge-score nltk sentencepiece
!pip install wandb tensorboard scikit-learn

In [ ]:
# Import required libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR

import transformers
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, 
    T5ForConditionalGeneration, T5Tokenizer,
    get_linear_schedule_with_warmup
)

import datasets
from datasets import load_dataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import re
import os
import json
import random
import logging
from tqdm.auto import tqdm
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass
from pathlib import Path

# Evaluation metrics
import evaluate
from rouge_score import rouge_scorer
import nltk
nltk.download('punkt')

# Set random seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 2. Configuration and Hyperparameters

In [ ]:
@dataclass
class Config:
    # Model configuration
    model_name: str = "Salesforce/codet5-base"
    max_source_length: int = 512
    max_target_length: int = 512
    
    # Training configuration
    batch_size: int = 8
    learning_rate: float = 5e-5
    num_epochs: int = 3
    warmup_steps: int = 1000
    weight_decay: float = 0.01
    gradient_accumulation_steps: int = 4
    max_grad_norm: float = 1.0
    
    # Dataset configuration
    dataset_name: str = "code_x_glue_cc_code_refinement"
    train_subset_size: Optional[int] = None  # Use None for full dataset
    val_subset_size: Optional[int] = None
    
    # Evaluation configuration
    eval_steps: int = 500
    save_steps: int = 1000
    logging_steps: int = 100
    
    # Output configuration
    output_dir: str = "./code_refactoring_model"
    save_total_limit: int = 3
    
    # Generation configuration
    num_beams: int = 5
    early_stopping: bool = True
    
config = Config()
print(f"Configuration: {config}")

## 3. Data Loader Class

In [ ]:
class CodeRefinementDataLoader:
    """
    Data loader for the CodeXGLUE code refinement dataset.
    Handles loading, filtering, and batching of code samples.
    """
    
    def __init__(self, config: Config):
        self.config = config
        self.dataset = None
        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None
        
    def load_dataset(self):
        """Load the CodeXGLUE code refinement dataset."""
        logger.info(f"Loading dataset: {self.config.dataset_name}")
        
        try:
            # Load the dataset from Hugging Face
            self.dataset = load_dataset(self.config.dataset_name)
            
            # Extract train, validation, and test splits
            self.train_dataset = self.dataset['train']
            self.val_dataset = self.dataset['validation']
            self.test_dataset = self.dataset['test']
            
            logger.info(f"Dataset loaded successfully:")
            logger.info(f"  Train samples: {len(self.train_dataset)}")
            logger.info(f"  Validation samples: {len(self.val_dataset)}")
            logger.info(f"  Test samples: {len(self.test_dataset)}")
            
            # Apply subset if specified
            if self.config.train_subset_size:
                self.train_dataset = self.train_dataset.select(range(self.config.train_subset_size))
                logger.info(f"  Train subset: {len(self.train_dataset)}")
                
            if self.config.val_subset_size:
                self.val_dataset = self.val_dataset.select(range(self.config.val_subset_size))
                logger.info(f"  Validation subset: {len(self.val_dataset)}")
                
        except Exception as e:
            logger.error(f"Error loading dataset: {e}")
            raise
    
    def get_sample(self, split: str = 'train', index: int = 0) -> Dict[str, str]:
        """Get a sample from the dataset for inspection."""
        if split == 'train':
            return self.train_dataset[index]
        elif split == 'validation':
            return self.val_dataset[index]
        elif split == 'test':
            return self.test_dataset[index]
        else:
            raise ValueError(f"Invalid split: {split}")
    
    def get_data_stats(self) -> Dict[str, Any]:
        """Get statistics about the dataset."""
        stats = {
            'train_size': len(self.train_dataset),
            'val_size': len(self.val_dataset),
            'test_size': len(self.test_dataset),
        }
        
        # Calculate average lengths
        sample_size = min(1000, len(self.train_dataset))
        samples = self.train_dataset.select(range(sample_size))
        
        buggy_lengths = [len(sample['buggy'].split()) for sample in samples]
        fixed_lengths = [len(sample['fixed'].split()) for sample in samples]
        
        stats.update({
            'avg_buggy_length': np.mean(buggy_lengths),
            'avg_fixed_length': np.mean(fixed_lengths),
            'max_buggy_length': np.max(buggy_lengths),
            'max_fixed_length': np.max(fixed_lengths),
        })
        
        return stats
    
    def create_torch_dataset(self, split: str = 'train'):
        """Create a PyTorch dataset for the given split."""
        if split == 'train':
            return self.train_dataset
        elif split == 'validation':
            return self.val_dataset
        elif split == 'test':
            return self.test_dataset
        else:
            raise ValueError(f"Invalid split: {split}")

# Test the data loader
data_loader = CodeRefinementDataLoader(config)
data_loader.load_dataset()

In [ ]:
# Explore the dataset
print("Dataset Statistics:")
stats = data_loader.get_data_stats()
for key, value in stats.items():
    print(f"  {key}: {value}")

print("\nSample Data:")
sample = data_loader.get_sample('train', 0)
print(f"Buggy Code:\n{sample['buggy']}")
print(f"\nFixed Code:\n{sample['fixed']}")

## 4. Tokenizer Class

In [ ]:
class CodeT5Tokenizer:
    """
    Tokenizer wrapper for CodeT5 model with custom preprocessing for code.
    """
    
    def __init__(self, config: Config):
        self.config = config
        self.tokenizer = None
        self.special_tokens = {
            'pad_token': '<pad>',
            'eos_token': '</s>',
            'bos_token': '<s>',
            'unk_token': '<unk>',
            'mask_token': '<mask>'
        }
        
    def load_tokenizer(self):
        """Load the CodeT5 tokenizer."""
        logger.info(f"Loading tokenizer: {self.config.model_name}")
        
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(
                self.config.model_name,
                cache_dir=None,
                use_fast=True
            )
            
            # Add special tokens if not present
            special_tokens_dict = {}
            for key, token in self.special_tokens.items():
                if getattr(self.tokenizer, key, None) is None:
                    special_tokens_dict[key] = token
            
            if special_tokens_dict:
                self.tokenizer.add_special_tokens(special_tokens_dict)
                logger.info(f"Added special tokens: {special_tokens_dict}")
            
            logger.info(f"Tokenizer loaded successfully")
            logger.info(f"  Vocabulary size: {len(self.tokenizer)}")
            logger.info(f"  Max length: {self.tokenizer.model_max_length}")
            
        except Exception as e:
            logger.error(f"Error loading tokenizer: {e}")
            raise
    
    def encode_batch(self, texts: List[str], max_length: int = None, 
                    padding: str = 'max_length', truncation: bool = True) -> Dict[str, torch.Tensor]:
        """Encode a batch of texts."""
        if max_length is None:
            max_length = self.config.max_source_length
        
        encoding = self.tokenizer(
            texts,
            max_length=max_length,
            padding=padding,
            truncation=truncation,
            return_tensors='pt'
        )
        
        return encoding
    
    def decode_batch(self, token_ids: torch.Tensor, skip_special_tokens: bool = True) -> List[str]:
        """Decode a batch of token IDs."""
        return self.tokenizer.batch_decode(
            token_ids,
            skip_special_tokens=skip_special_tokens
        )
    
    def tokenize_code_pair(self, buggy_code: str, fixed_code: str) -> Dict[str, torch.Tensor]:
        """Tokenize a pair of buggy and fixed code."""
        # Tokenize source (buggy) code
        source_encoding = self.encode_batch(
            [buggy_code],
            max_length=self.config.max_source_length
        )
        
        # Tokenize target (fixed) code
        target_encoding = self.encode_batch(
            [fixed_code],
            max_length=self.config.max_target_length
        )
        
        return {
            'input_ids': source_encoding['input_ids'].squeeze(0),
            'attention_mask': source_encoding['attention_mask'].squeeze(0),
            'labels': target_encoding['input_ids'].squeeze(0)
        }
    
    def get_vocab_size(self) -> int:
        """Get the vocabulary size."""
        return len(self.tokenizer)

# Test the tokenizer
tokenizer = CodeT5Tokenizer(config)
tokenizer.load_tokenizer()

In [ ]:
# Test tokenization
sample = data_loader.get_sample('train', 0)
tokenized = tokenizer.tokenize_code_pair(sample['buggy'], sample['fixed'])

print(f"Input IDs shape: {tokenized['input_ids'].shape}")
print(f"Attention mask shape: {tokenized['attention_mask'].shape}")
print(f"Labels shape: {tokenized['labels'].shape}")

# Decode to verify
decoded_input = tokenizer.decode_batch(tokenized['input_ids'].unsqueeze(0))
decoded_labels = tokenizer.decode_batch(tokenized['labels'].unsqueeze(0))

print(f"\nDecoded input: {decoded_input[0][:200]}...")
print(f"Decoded labels: {decoded_labels[0][:200]}...")

## 5. Preprocessor Class

In [ ]:
class CodePreprocessor:
    """
    Preprocessor for cleaning and formatting code samples.
    """
    
    def __init__(self, config: Config):
        self.config = config
        self.max_line_length = 100
        
    def normalize_whitespace(self, code: str) -> str:
        """Normalize whitespace in code."""
        # Replace multiple spaces with single space
        code = re.sub(r' +', ' ', code)
        # Replace multiple newlines with single newline
        code = re.sub(r'\n+', '\n', code)
        # Remove trailing whitespace
        code = '\n'.join(line.rstrip() for line in code.split('\n'))
        return code.strip()
    
    def remove_comments(self, code: str) -> str:
        """Remove comments from code (optional preprocessing)."""
        # Remove single-line comments
        code = re.sub(r'//.*$', '', code, flags=re.MULTILINE)
        # Remove multi-line comments
        code = re.sub(r'/\*.*?\*/', '', code, flags=re.DOTALL)
        return code
    
    def format_code(self, code: str) -> str:
        """Format code for better tokenization."""
        # Add spaces around operators
        code = re.sub(r'([+\-*/=<>!&|])([^=])', r'\1 \2', code)
        code = re.sub(r'([^=+\-*/=<>!&|])([+\-*/=<>!&|])', r'\1 \2', code)
        
        # Add spaces around parentheses and brackets
        code = re.sub(r'([\(\[\{])', r' \1 ', code)
        code = re.sub(r'([\)\]\}])', r' \1 ', code)
        
        # Add spaces around commas and semicolons
        code = re.sub(r'([,;])', r'\1 ', code)
        
        # Normalize whitespace after formatting
        code = self.normalize_whitespace(code)
        
        return code
    
    def add_task_prefix(self, code: str, task_type: str = "refactor") -> str:
        """Add task-specific prefix to code."""
        prefixes = {
            "refactor": "Refactor this code:",
            "debug": "Debug this code:",
            "fix": "Fix this code:"
        }
        prefix = prefixes.get(task_type, "Refactor this code:")
        return f"{prefix} {code}"
    
    def preprocess_code_pair(self, buggy_code: str, fixed_code: str, 
                           remove_comments: bool = False, 
                           add_prefix: bool = True) -> Tuple[str, str]:
        """Preprocess a pair of buggy and fixed code."""
        # Process buggy code
        processed_buggy = self.normalize_whitespace(buggy_code)
        if remove_comments:
            processed_buggy = self.remove_comments(processed_buggy)
        processed_buggy = self.format_code(processed_buggy)
        
        if add_prefix:
            processed_buggy = self.add_task_prefix(processed_buggy)
        
        # Process fixed code
        processed_fixed = self.normalize_whitespace(fixed_code)
        if remove_comments:
            processed_fixed = self.remove_comments(processed_fixed)
        processed_fixed = self.format_code(processed_fixed)
        
        return processed_buggy, processed_fixed
    
    def filter_by_length(self, code: str, max_tokens: int = None) -> bool:
        """Filter code by token length."""
        if max_tokens is None:
            max_tokens = self.config.max_source_length
        
        token_count = len(code.split())
        return token_count <= max_tokens
    
    def get_code_complexity(self, code: str) -> Dict[str, int]:
        """Calculate basic code complexity metrics."""
        lines = code.split('\n')
        
        metrics = {
            'line_count': len(lines),
            'char_count': len(code),
            'word_count': len(code.split()),
            'max_line_length': max(len(line) for line in lines) if lines else 0,
            'avg_line_length': sum(len(line) for line in lines) / len(lines) if lines else 0,
            'empty_lines': sum(1 for line in lines if not line.strip()),
            'indent_levels': len(set(len(line) - len(line.lstrip()) for line in lines if line.strip()))
        }
        
        return metrics

# Test the preprocessor
preprocessor = CodePreprocessor(config)

sample = data_loader.get_sample('train', 0)
processed_buggy, processed_fixed = preprocessor.preprocess_code_pair(
    sample['buggy'], sample['fixed']
)

print("Original buggy code:")
print(sample['buggy'][:200] + "...")
print("\nProcessed buggy code:")
print(processed_buggy[:200] + "...")

print("\nCode complexity metrics:")
complexity = preprocessor.get_code_complexity(processed_buggy)
for key, value in complexity.items():
    print(f"  {key}: {value}")

## 6. Model Architecture Class

In [ ]:
class CodeT5ModelArchitecture(nn.Module):
    """
    CodeT5 model architecture for code refactoring.
    """
    
    def __init__(self, config: Config, tokenizer: CodeT5Tokenizer):
        super().__init__()
        self.config = config
        self.tokenizer = tokenizer
        
        # Load the pre-trained CodeT5 model
        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            config.model_name,
            cache_dir=None
        )
        
        # Resize token embeddings if tokenizer was extended
        if len(tokenizer.tokenizer) > self.model.config.vocab_size:
            self.model.resize_token_embeddings(len(tokenizer.tokenizer))
            logger.info(f"Resized token embeddings to {len(tokenizer.tokenizer)}")
        
        # Additional layers for code-specific features
        self.dropout = nn.Dropout(0.1)
        
        # Code quality prediction head (optional)
        self.quality_head = nn.Linear(self.model.config.d_model, 1)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        """Initialize additional weights."""
        nn.init.xavier_uniform_(self.quality_head.weight)
        nn.init.zeros_(self.quality_head.bias)
    
    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, 
                labels: torch.Tensor = None, return_dict: bool = True) -> Dict[str, torch.Tensor]:
        """Forward pass of the model."""
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            return_dict=return_dict
        )
        
        if return_dict:
            result = {
                'loss': outputs.loss,
                'logits': outputs.logits,
                'past_key_values': outputs.past_key_values,
                'decoder_hidden_states': outputs.decoder_hidden_states,
                'decoder_attentions': outputs.decoder_attentions,
                'cross_attentions': outputs.cross_attentions,
                'encoder_last_hidden_state': outputs.encoder_last_hidden_state,
                'encoder_hidden_states': outputs.encoder_hidden_states,
                'encoder_attentions': outputs.encoder_attentions
            }
            return result
        
        return outputs
    
    def generate(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, 
                max_length: int = None, num_beams: int = None, 
                early_stopping: bool = None, **kwargs) -> torch.Tensor:
        """Generate code using the model."""
        if max_length is None:
            max_length = self.config.max_target_length
        if num_beams is None:
            num_beams = self.config.num_beams
        if early_stopping is None:
            early_stopping = self.config.early_stopping
        
        with torch.no_grad():
            outputs = self.model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=max_length,
                num_beams=num_beams,
                early_stopping=early_stopping,
                pad_token_id=self.tokenizer.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.tokenizer.eos_token_id,
                **kwargs
            )
        
        return outputs
    
    def get_encoder_outputs(self, input_ids: torch.Tensor, 
                           attention_mask: torch.Tensor) -> torch.Tensor:
        """Get encoder outputs for analysis."""
        with torch.no_grad():
            outputs = self.model.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_dict=True
            )
        return outputs.last_hidden_state
    
    def predict_quality(self, input_ids: torch.Tensor, 
                       attention_mask: torch.Tensor) -> torch.Tensor:
        """Predict code quality score."""
        encoder_outputs = self.get_encoder_outputs(input_ids, attention_mask)
        # Use mean pooling
        pooled_output = encoder_outputs.mean(dim=1)
        quality_score = self.quality_head(pooled_output)
        return torch.sigmoid(quality_score)
    
    def count_parameters(self) -> Dict[str, int]:
        """Count model parameters."""
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        
        return {
            'total_parameters': total_params,
            'trainable_parameters': trainable_params,
            'non_trainable_parameters': total_params - trainable_params
        }
    
    def freeze_encoder(self):
        """Freeze encoder parameters."""
        for param in self.model.encoder.parameters():
            param.requires_grad = False
        logger.info("Encoder parameters frozen")
    
    def unfreeze_encoder(self):
        """Unfreeze encoder parameters."""
        for param in self.model.encoder.parameters():
            param.requires_grad = True
        logger.info("Encoder parameters unfrozen")

# Initialize the model
model = CodeT5ModelArchitecture(config, tokenizer)
model = model.to(device)

# Print model statistics
param_stats = model.count_parameters()
print("Model Parameter Statistics:")
for key, value in param_stats.items():
    print(f"  {key}: {value:,}")

print(f"\nModel size: {param_stats['total_parameters'] * 4 / (1024**2):.2f} MB")

## 7. Training Class

In [ ]:
class CodeRefactoringTrainer:
    """
    Training class for code refactoring model.
    """
    
    def __init__(self, model: CodeT5ModelArchitecture, tokenizer: CodeT5Tokenizer, 
                 preprocessor: CodePreprocessor, config: Config):
        self.model = model
        self.tokenizer = tokenizer
        self.preprocessor = preprocessor
        self.config = config
        
        # Initialize optimizer and scheduler
        self.optimizer = None
        self.scheduler = None
        
        # Training statistics
        self.train_losses = []
        self.val_losses = []
        self.learning_rates = []
        
        # Best model tracking
        self.best_val_loss = float('inf')
        self.best_model_state = None
        
        # Create output directory
        os.makedirs(config.output_dir, exist_ok=True)
    
    def create_dataset(self, dataset, split: str = 'train'):
        """Create a PyTorch dataset from HuggingFace dataset."""
        class CodeDataset(Dataset):
            def __init__(self, hf_dataset, tokenizer, preprocessor, config):
                self.dataset = hf_dataset
                self.tokenizer = tokenizer
                self.preprocessor = preprocessor
                self.config = config
            
            def __len__(self):
                return len(self.dataset)
            
            def __getitem__(self, idx):
                item = self.dataset[idx]
                
                # Preprocess the code pair
                buggy_code, fixed_code = self.preprocessor.preprocess_code_pair(
                    item['buggy'], item['fixed']
                )
                
                # Tokenize
                tokenized = self.tokenizer.tokenize_code_pair(buggy_code, fixed_code)
                
                return {
                    'input_ids': tokenized['input_ids'],
                    'attention_mask': tokenized['attention_mask'],
                    'labels': tokenized['labels']
                }
        
        return CodeDataset(dataset, self.tokenizer, self.preprocessor, self.config)
    
    def collate_fn(self, batch):
        """Collate function for batching."""
        input_ids = torch.stack([item['input_ids'] for item in batch])
        attention_mask = torch.stack([item['attention_mask'] for item in batch])
        labels = torch.stack([item['labels'] for item in batch])
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels
        }
    
    def setup_training(self, train_dataset, val_dataset):
        """Setup training components."""
        # Create data loaders
        self.train_loader = DataLoader(
            train_dataset,
            batch_size=self.config.batch_size,
            shuffle=True,
            collate_fn=self.collate_fn,
            num_workers=0  # Use 0 for Kaggle
        )
        
        self.val_loader = DataLoader(
            val_dataset,
            batch_size=self.config.batch_size,
            shuffle=False,
            collate_fn=self.collate_fn,
            num_workers=0
        )
        
        # Setup optimizer
        no_decay = ['bias', 'LayerNorm.weight']
        optimizer_grouped_parameters = [
            {
                'params': [p for n, p in self.model.named_parameters() 
                          if not any(nd in n for nd in no_decay)],
                'weight_decay': self.config.weight_decay,
            },
            {
                'params': [p for n, p in self.model.named_parameters() 
                          if any(nd in n for nd in no_decay)],
                'weight_decay': 0.0,
            },
        ]
        
        self.optimizer = AdamW(
            optimizer_grouped_parameters,
            lr=self.config.learning_rate,
            eps=1e-8
        )
        
        # Setup scheduler
        total_steps = len(self.train_loader) * self.config.num_epochs
        self.scheduler = get_linear_schedule_with_warmup(
            self.optimizer,
            num_warmup_steps=self.config.warmup_steps,
            num_training_steps=total_steps
        )
        
        logger.info(f"Training setup complete:")
        logger.info(f"  Training samples: {len(train_dataset)}")
        logger.info(f"  Validation samples: {len(val_dataset)}")
        logger.info(f"  Batch size: {self.config.batch_size}")
        logger.info(f"  Total training steps: {total_steps}")
    
    def train_epoch(self, epoch: int) -> float:
        """Train for one epoch."""
        self.model.train()
        total_loss = 0
        num_batches = 0
        
        progress_bar = tqdm(self.train_loader, desc=f"Epoch {epoch+1}/{self.config.num_epochs}")
        
        for step, batch in enumerate(progress_bar):
            # Move batch to device
            batch = {k: v.to(device) for k, v in batch.items()}
            
            # Forward pass
            outputs = self.model(**batch)
            loss = outputs['loss']
            
            # Backward pass
            if self.config.gradient_accumulation_steps > 1:
                loss = loss / self.config.gradient_accumulation_steps
            
            loss.backward()
            
            # Update weights
            if (step + 1) % self.config.gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.max_grad_norm)
                self.optimizer.step()
                self.scheduler.step()
                self.optimizer.zero_grad()
            
            # Update statistics
            total_loss += loss.item()
            num_batches += 1
            
            # Update progress bar
            current_lr = self.scheduler.get_last_lr()[0]
            progress_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'lr': f'{current_lr:.2e}'
            })
            
            # Log training statistics
            if (step + 1) % self.config.logging_steps == 0:
                avg_loss = total_loss / num_batches
                self.train_losses.append(avg_loss)
                self.learning_rates.append(current_lr)
                
                logger.info(f"Step {step+1}: loss={avg_loss:.4f}, lr={current_lr:.2e}")
        
        return total_loss / num_batches
    
    def validate(self) -> float:
        """Validate the model."""
        self.model.eval()
        total_loss = 0
        num_batches = 0
        
        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc="Validating"):
                batch = {k: v.to(device) for k, v in batch.items()}
                
                outputs = self.model(**batch)
                loss = outputs['loss']
                
                total_loss += loss.item()
                num_batches += 1
        
        avg_loss = total_loss / num_batches
        self.val_losses.append(avg_loss)
        
        return avg_loss
    
    def save_model(self, epoch: int, val_loss: float):
        """Save model checkpoint."""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'val_loss': val_loss,
            'config': self.config
        }
        
        checkpoint_path = os.path.join(self.config.output_dir, f'checkpoint_epoch_{epoch}.pt')
        torch.save(checkpoint, checkpoint_path)
        
        # Save best model
        if val_loss < self.best_val_loss:
            self.best_val_loss = val_loss
            self.best_model_state = self.model.state_dict().copy()
            
            best_model_path = os.path.join(self.config.output_dir, 'best_model.pt')
            torch.save(checkpoint, best_model_path)
            logger.info(f"New best model saved with validation loss: {val_loss:.4f}")
    
    def train(self, train_dataset, val_dataset):
        """Main training loop."""
        logger.info("Starting training...")
        
        # Setup training
        self.setup_training(train_dataset, val_dataset)
        
        # Training loop
        for epoch in range(self.config.num_epochs):
            # Train epoch
            train_loss = self.train_epoch(epoch)
            
            # Validate
            val_loss = self.validate()
            
            # Save checkpoint
            if (epoch + 1) % (self.config.save_steps // len(self.train_loader)) == 0:
                self.save_model(epoch, val_loss)
            
            logger.info(f"Epoch {epoch+1}/{self.config.num_epochs}:")
            logger.info(f"  Train loss: {train_loss:.4f}")
            logger.info(f"  Val loss: {val_loss:.4f}")
            logger.info(f"  Best val loss: {self.best_val_loss:.4f}")
        
        logger.info("Training completed!")
        
        # Load best model
        if self.best_model_state is not None:
            self.model.load_state_dict(self.best_model_state)
            logger.info("Best model loaded")
    
    def plot_training_curves(self):
        """Plot training curves."""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        
        # Loss curves
        ax1.plot(self.train_losses, label='Training Loss')
        ax1.plot(range(0, len(self.train_losses), len(self.train_losses)//len(self.val_losses)), 
                self.val_losses, label='Validation Loss')
        ax1.set_xlabel('Training Steps')
        ax1.set_ylabel('Loss')
        ax1.set_title('Training and Validation Loss')
        ax1.legend()
        ax1.grid(True)
        
        # Learning rate curve
        ax2.plot(self.learning_rates)
        ax2.set_xlabel('Training Steps')
        ax2.set_ylabel('Learning Rate')
        ax2.set_title('Learning Rate Schedule')
        ax2.grid(True)
        
        plt.tight_layout()
        plt.savefig(os.path.join(self.config.output_dir, 'training_curves.png'))
        plt.show()

# Initialize trainer
trainer = CodeRefactoringTrainer(model, tokenizer, preprocessor, config)
logger.info("Trainer initialized successfully")

## 8. Evaluation Class

In [ ]:
class CodeRefactoringEvaluator:
    """
    Evaluation class for code refactoring model.
    """
    
    def __init__(self, model: CodeT5ModelArchitecture, tokenizer: CodeT5Tokenizer, 
                 preprocessor: CodePreprocessor, config: Config):
        self.model = model
        self.tokenizer = tokenizer
        self.preprocessor = preprocessor
        self.config = config
        
        # Initialize evaluation metrics
        self.bleu_metric = evaluate.load('bleu')
        self.rouge_scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        
    def compute_bleu(self, predictions: List[str], references: List[str]) -> Dict[str, float]:
        """Compute BLEU score."""
        # Tokenize predictions and references
        tokenized_predictions = [pred.split() for pred in predictions]
        tokenized_references = [[ref.split()] for ref in references]
        
        bleu_score = self.bleu_metric.compute(
            predictions=tokenized_predictions,
            references=tokenized_references
        )
        
        return bleu_score
    
    def compute_rouge(self, predictions: List[str], references: List[str]) -> Dict[str, float]:
        """Compute ROUGE scores."""
        rouge_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
        
        for pred, ref in zip(predictions, references):
            scores = self.rouge_scorer.score(ref, pred)
            rouge_scores['rouge1'].append(scores['rouge1'].fmeasure)
            rouge_scores['rouge2'].append(scores['rouge2'].fmeasure)
            rouge_scores['rougeL'].append(scores['rougeL'].fmeasure)
        
        # Average scores
        avg_rouge_scores = {
            'rouge1': np.mean(rouge_scores['rouge1']),
            'rouge2': np.mean(rouge_scores['rouge2']),
            'rougeL': np.mean(rouge_scores['rougeL'])
        }
        
        return avg_rouge_scores
    
    def compute_exact_match(self, predictions: List[str], references: List[str]) -> float:
        """Compute exact match accuracy."""
        exact_matches = sum(1 for pred, ref in zip(predictions, references) 
                           if pred.strip() == ref.strip())
        return exact_matches / len(predictions)
    
    def compute_code_bleu(self, predictions: List[str], references: List[str]) -> float:
        """Compute CodeBLEU score (simplified version)."""
        # This is a simplified version - in practice, you'd use the official CodeBLEU implementation
        # For now, we'll use a weighted combination of BLEU and syntactic similarity
        
        bleu_score = self.compute_bleu(predictions, references)['bleu']
        
        # Simple syntactic similarity (count of matching tokens)
        syntactic_scores = []
        for pred, ref in zip(predictions, references):
            pred_tokens = set(pred.split())
            ref_tokens = set(ref.split())
            if len(ref_tokens) > 0:
                similarity = len(pred_tokens & ref_tokens) / len(ref_tokens)
            else:
                similarity = 0.0
            syntactic_scores.append(similarity)
        
        syntactic_score = np.mean(syntactic_scores)
        
        # Weighted combination
        code_bleu = 0.7 * bleu_score + 0.3 * syntactic_score
        
        return code_bleu
    
    def evaluate_sample(self, buggy_code: str, expected_fixed: str) -> Dict[str, Any]:
        """Evaluate a single sample."""
        # Preprocess input
        processed_buggy, _ = self.preprocessor.preprocess_code_pair(buggy_code, expected_fixed)
        
        # Tokenize input
        inputs = self.tokenizer.encode_batch([processed_buggy])
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Generate prediction
        with torch.no_grad():
            outputs = self.model.generate(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask']
            )
        
        # Decode prediction
        predicted_code = self.tokenizer.decode_batch(outputs)[0]
        
        # Compute metrics
        bleu_score = self.compute_bleu([predicted_code], [expected_fixed])
        rouge_scores = self.compute_rouge([predicted_code], [expected_fixed])
        exact_match = self.compute_exact_match([predicted_code], [expected_fixed])
        code_bleu = self.compute_code_bleu([predicted_code], [expected_fixed])
        
        return {
            'buggy_code': buggy_code,
            'expected_fixed': expected_fixed,
            'predicted_code': predicted_code,
            'bleu': bleu_score['bleu'],
            'rouge1': rouge_scores['rouge1'],
            'rouge2': rouge_scores['rouge2'],
            'rougeL': rouge_scores['rougeL'],
            'exact_match': exact_match,
            'code_bleu': code_bleu
        }
    
    def evaluate_dataset(self, dataset, num_samples: int = None) -> Dict[str, float]:
        """Evaluate on a dataset."""
        if num_samples is None:
            num_samples = len(dataset)
        
        # Sample data
        if num_samples < len(dataset):
            indices = random.sample(range(len(dataset)), num_samples)
            samples = [dataset[i] for i in indices]
        else:
            samples = dataset
        
        predictions = []
        references = []
        
        logger.info(f"Evaluating {len(samples)} samples...")
        
        for sample in tqdm(samples, desc="Evaluating"):
            # Preprocess
            processed_buggy, processed_fixed = self.preprocessor.preprocess_code_pair(
                sample['buggy'], sample['fixed']
            )
            
            # Tokenize input
            inputs = self.tokenizer.encode_batch([processed_buggy])
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            # Generate prediction
            with torch.no_grad():
                outputs = self.model.generate(
                    input_ids=inputs['input_ids'],
                    attention_mask=inputs['attention_mask']
                )
            
            # Decode prediction
            predicted_code = self.tokenizer.decode_batch(outputs)[0]
            
            predictions.append(predicted_code)
            references.append(processed_fixed)
        
        # Compute metrics
        bleu_score = self.compute_bleu(predictions, references)
        rouge_scores = self.compute_rouge(predictions, references)
        exact_match = self.compute_exact_match(predictions, references)
        code_bleu = self.compute_code_bleu(predictions, references)
        
        evaluation_results = {
            'bleu': bleu_score['bleu'],
            'rouge1': rouge_scores['rouge1'],
            'rouge2': rouge_scores['rouge2'],
            'rougeL': rouge_scores['rougeL'],
            'exact_match': exact_match,
            'code_bleu': code_bleu,
            'num_samples': len(samples)
        }
        
        return evaluation_results
    
    def generate_evaluation_report(self, results: Dict[str, float]) -> str:
        """Generate a formatted evaluation report."""
        report = "\n" + "="*50 + "\n"
        report += "         EVALUATION REPORT\n"
        report += "="*50 + "\n"
        report += f"Number of samples: {results['num_samples']}\n"
        report += "-"*50 + "\n"
        report += f"BLEU Score:        {results['bleu']:.4f}\n"
        report += f"CodeBLEU Score:    {results['code_bleu']:.4f}\n"
        report += f"ROUGE-1:           {results['rouge1']:.4f}\n"
        report += f"ROUGE-2:           {results['rouge2']:.4f}\n"
        report += f"ROUGE-L:           {results['rougeL']:.4f}\n"
        report += f"Exact Match:       {results['exact_match']:.4f}\n"
        report += "="*50 + "\n"
        
        return report
    
    def save_evaluation_results(self, results: Dict[str, float], filepath: str):
        """Save evaluation results to file."""
        with open(filepath, 'w') as f:
            json.dump(results, f, indent=2)
        logger.info(f"Evaluation results saved to {filepath}")

# Initialize evaluator
evaluator = CodeRefactoringEvaluator(model, tokenizer, preprocessor, config)
logger.info("Evaluator initialized successfully")

## 9. Training Pipeline

In [ ]:
# Create datasets
train_dataset = trainer.create_dataset(data_loader.train_dataset, 'train')
val_dataset = trainer.create_dataset(data_loader.val_dataset, 'validation')

logger.info(f"Created datasets:")
logger.info(f"  Training samples: {len(train_dataset)}")
logger.info(f"  Validation samples: {len(val_dataset)}")

# Test a single batch
sample_batch = trainer.collate_fn([train_dataset[i] for i in range(2)])
print(f"Sample batch shapes:")
for key, value in sample_batch.items():
    print(f"  {key}: {value.shape}")

In [ ]:
# Start training
logger.info("Starting training process...")
trainer.train(train_dataset, val_dataset)

# Plot training curves
trainer.plot_training_curves()

## 10. Model Evaluation

In [ ]:
# Evaluate on test set
logger.info("Evaluating model on test set...")
test_results = evaluator.evaluate_dataset(data_loader.test_dataset, num_samples=100)

# Print evaluation report
report = evaluator.generate_evaluation_report(test_results)
print(report)

# Save results
evaluator.save_evaluation_results(
    test_results, 
    os.path.join(config.output_dir, 'test_evaluation_results.json')
)

## 11. Sample Predictions

In [ ]:
# Test on sample predictions
def test_sample_predictions(num_samples=5):
    """Test model predictions on sample data."""
    logger.info(f"Testing {num_samples} sample predictions...")
    
    for i in range(num_samples):
        sample = data_loader.get_sample('test', i)
        result = evaluator.evaluate_sample(sample['buggy'], sample['fixed'])
        
        print(f"\n{'='*60}")
        print(f"SAMPLE {i+1}")
        print(f"{'='*60}")
        print(f"\nBUGGY CODE:\n{result['buggy_code']}")
        print(f"\nEXPECTED FIXED:\n{result['expected_fixed']}")
        print(f"\nPREDICTED FIXED:\n{result['predicted_code']}")
        print(f"\nMETRICS:")
        print(f"  BLEU: {result['bleu']:.4f}")
        print(f"  CodeBLEU: {result['code_bleu']:.4f}")
        print(f"  ROUGE-1: {result['rouge1']:.4f}")
        print(f"  ROUGE-L: {result['rougeL']:.4f}")
        print(f"  Exact Match: {result['exact_match']:.4f}")

# Run sample predictions
test_sample_predictions()

## 12. Model Inference Function

In [ ]:
def refactor_code(buggy_code: str, model: CodeT5ModelArchitecture, 
                 tokenizer: CodeT5Tokenizer, preprocessor: CodePreprocessor) -> str:
    """
    Refactor buggy code using the trained model.
    
    Args:
        buggy_code: The buggy code to refactor
        model: Trained CodeT5 model
        tokenizer: CodeT5 tokenizer
        preprocessor: Code preprocessor
    
    Returns:
        Refactored code
    """
    # Preprocess the input
    processed_code = preprocessor.add_task_prefix(buggy_code)
    processed_code = preprocessor.normalize_whitespace(processed_code)
    
    # Tokenize
    inputs = tokenizer.encode_batch([processed_code])
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Generate
    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=config.max_target_length,
            num_beams=config.num_beams,
            early_stopping=config.early_stopping,
            temperature=0.7,
            do_sample=False
        )
    
    # Decode
    refactored_code = tokenizer.decode_batch(outputs)[0]
    
    return refactored_code

# Test the inference function
test_buggy_code = """
def calculate_average(numbers):
    sum = 0
    for i in range(len(numbers)):
        sum += numbers[i]
    return sum / len(numbers)
"""

refactored = refactor_code(test_buggy_code, model, tokenizer, preprocessor)
print("Original buggy code:")
print(test_buggy_code)
print("\nRefactored code:")
print(refactored)

## 13. Model Export and Saving

In [ ]:
# Save the final model and tokenizer
def save_model_for_inference(model, tokenizer, config, save_path="./final_model"):
    """Save model and tokenizer for inference."""
    os.makedirs(save_path, exist_ok=True)
    
    # Save model
    model.model.save_pretrained(save_path)
    tokenizer.tokenizer.save_pretrained(save_path)
    
    # Save config
    config_dict = {
        'model_name': config.model_name,
        'max_source_length': config.max_source_length,
        'max_target_length': config.max_target_length,
        'num_beams': config.num_beams,
        'early_stopping': config.early_stopping
    }
    
    with open(os.path.join(save_path, 'config.json'), 'w') as f:
        json.dump(config_dict, f, indent=2)
    
    logger.info(f"Model saved to {save_path}")

# Save the final model
save_model_for_inference(model, tokenizer, config)

# Create a summary of the training process
training_summary = {
    'dataset': config.dataset_name,
    'model': config.model_name,
    'training_samples': len(train_dataset),
    'validation_samples': len(val_dataset),
    'num_epochs': config.num_epochs,
    'batch_size': config.batch_size,
    'learning_rate': config.learning_rate,
    'best_val_loss': trainer.best_val_loss,
    'final_test_results': test_results
}

with open(os.path.join(config.output_dir, 'training_summary.json'), 'w') as f:
    json.dump(training_summary, f, indent=2)

print("Training Summary:")
print(json.dumps(training_summary, indent=2))

## 14. Conclusion and Usage Instructions

This notebook provides a complete pipeline for training a CodeT5 model on code refactoring tasks. Here's how to use it:

### Key Components:

1. **Data Loader**: Loads and processes the CodeXGLUE code refinement dataset
2. **Tokenizer**: Handles tokenization of code with CodeT5 tokenizer
3. **Preprocessor**: Cleans and formats code for better training
4. **Model Architecture**: CodeT5 model with custom heads for refactoring
5. **Training**: Complete training loop with optimization and scheduling
6. **Evaluation**: Comprehensive evaluation with BLEU, ROUGE, and CodeBLEU metrics

### Usage:

1. **For Training**: Run all cells in sequence to train the model
2. **For Inference**: Use the `refactor_code()` function with your trained model
3. **For Evaluation**: Use the `CodeRefactoringEvaluator` class to assess performance

### Model Performance:

The model is evaluated using:
- **BLEU Score**: Measures n-gram overlap between predicted and reference code
- **CodeBLEU**: Code-specific BLEU that considers syntactic similarity
- **ROUGE**: Measures recall-oriented overlap
- **Exact Match**: Percentage of perfectly matching predictions

### Customization:

You can customize the model by:
- Adjusting hyperparameters in the `Config` class
- Modifying preprocessing steps in `CodePreprocessor`
- Adding custom loss functions or regularization
- Implementing domain-specific evaluation metrics

### Next Steps:

1. Fine-tune hyperparameters based on validation performance
2. Implement more sophisticated preprocessing for different programming languages
3. Add support for multi-language code refactoring
4. Implement advanced evaluation metrics specific to code quality
5. Deploy the model for real-time code refactoring applications

The trained model can be used to automatically refactor buggy code, improving code quality and maintainability.